In [1]:
import argparse
import os.path as op
from stress_risk.utils.data import Subject
from nilearn import surface
import nibabel as nb
from stress_risk.fmri_analysis.encoding_model.fit_nprf import get_key_target_dir
from tqdm import tqdm
from nipype.interfaces.freesurfer import SurfaceTransform


/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/input_data/__init__.py:27: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  warnings.warn(message, FutureWarning)


231121-15:14:11,631 nipype.utils WARNING:
	 A newer version (1.8.4) of nipy/nipype is available. You are using 1.8.3


In [8]:
bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'
natural_space=False 
smoothed = True
denoise = False
subject = 26
session=1

_, target_dir = get_key_target_dir(f'{int(subject):02d}', session, bids_folder, smoothed, denoise=denoise, pca_confounds=False, retroicor=False, natural_space=natural_space) 

In [9]:

def transform_fsaverage(in_file, fs_hemi, source_subject, bids_folder):

        subjects_dir = op.join(bids_folder, 'derivatives', 'freesurfer')

        sxfm = SurfaceTransform(subjects_dir=subjects_dir)
        sxfm.inputs.source_file = in_file
        sxfm.inputs.out_file = in_file.replace('fsnative', 'fsaverage')
        sxfm.inputs.source_subject = source_subject
        sxfm.inputs.target_subject = 'fsaverage'
        sxfm.inputs.hemi = fs_hemi

        r = sxfm.run()
        return r


In [ ]:
par= 'mu' #'r2'
hemi = 'L'
target_fn =  op.join(target_dir, f'sub-{subject}_ses-{session}_desc-{par}.optim.nilearn_space-fsnative_hemi-{hemi}.func.gii')
fs_hemi = 'lh' if hemi == 'L' else 'rh'

transform_fsaverage(target_fn, fs_hemi, f'sub-{subject}', bids_folder)

In [14]:
from sample_nPRFparams_to_surf import main as sample_nPRFparams_to_surf
from os import listdir

subList = [int(f[4:6]) for f in listdir(op.join(bids_folder, 'derivatives','encoding_model.denoise.retroicor.smoothed')) if f[0:3] == 'sub']

retroicor = True
smoothed = True
denoise = True
session = 1

probLits = []
for subject in subList:
    try: 
        sample_nPRFparams_to_surf(subject, session, bids_folder, smoothed, denoise, retroicor,natural_space=False )
    except:
        probLits.append(subject)



Writing to /Volumes/mrenkeED/data/ds-stressrisk/derivatives/encoding_model.denoise.retroicor.smoothed/sub-01/ses-1/func
231122-09:19:01,273 nipype.interface INFO:
	 stdout 2023-11-22T09:19:01.273097:
231122-09:19:01,275 nipype.interface INFO:
	 stdout 2023-11-22T09:19:01.273097:7.2.0
231122-09:19:01,276 nipype.interface INFO:
	 stdout 2023-11-22T09:19:01.273097:
231122-09:19:01,276 nipype.interface INFO:
	 stdout 2023-11-22T09:19:01.273097:setenv SUBJECTS_DIR /Volumes/mrenkeED/data/ds-stressrisk/derivatives/freesurfer
231122-09:19:01,276 nipype.interface INFO:
	 stdout 2023-11-22T09:19:01.273097:cd /Users/mrenke/git/stress_risk/stress_risk/fmri_analysis/surface
231122-09:19:01,276 nipype.interface INFO:
	 stdout 2023-11-22T09:19:01.273097:mri_surf2surf --hemi lh --tval /Volumes/mrenkeED/data/ds-stressrisk/derivatives/encoding_model.denoise.retroicor.smoothed/sub-01/ses-1/func/sub-1_ses-1_desc-mu.optim.nilearn_space-fsaverage_hemi-L.func.gii --sval /Volumes/mrenkeED/data/ds-stressrisk/d

In [12]:
probLits

[1,
 2,
 3,
 4,
 5,
 9,
 14,
 16,
 17,
 22,
 34,
 35,
 36,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 57,
 58,
 59,
 61]

In [13]:
len(probLits)

35